# Experiment 4 — Ordinal Regression / Ordinal Loss

**Hypothesis:**

Standard cross-entropy treats Normal, Early, and Advanced as independent categorical classes and does not explicitly model their ordered relationship. An ordinal learning objective may better represent the severity structure of glaucoma and reduce severe ordinal misclassifications, potentially improving QWK and Early-class performance.

**Controlled change:**

Replace the standard three-class cross-entropy objective with a two-threshold ordinal regression formulation.

Everything else follows Experiment 1. Checkpoints are selected by lowest validation ordinal loss, and the test set is never evaluated.

In [ ]:
import gc, json, os, sys, time
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists(): PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists(): raise RuntimeError('Could not find src')
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from datasets import load_dataset
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
    classification_report, cohen_kappa_score, confusion_matrix,
    precision_recall_fscore_support)
from torch.utils.data import DataLoader
from torchvision import transforms
from src.datasets.glaucoma_dataset import GlaucomaDataset, IDX_TO_CLASS
from src.models.resnet50 import ResNet50Classifier
from src.training.config import (CHECKPOINT_DIR, LEARNING_RATE, NUM_CLASSES, NUM_EPOCHS,
    RESULTS_FIGURES_DIR, RESULTS_METRICS_DIR, SEED, WEIGHT_DECAY, set_seed)

if not torch.cuda.is_available(): raise RuntimeError('CUDA is not available')
DEVICE = torch.device('cuda:0')
set_seed(SEED)
sns.set_theme(style='whitegrid')
print('Project root:', PROJECT_ROOT)
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))

## Configuration

Experiment 4 starts independently from torchvision's ImageNet-pretrained ResNet-50 weights. No Experiment 1, 2, or 3 checkpoint is loaded. Resolution, data pipeline, optimizer, learning rate, weight decay, AMP, epoch limit, patience, checkpoint rule, and seed match Experiment 1.

In [ ]:
EXPERIMENT_NAME = 'EXPERIMENT_4_ORDINAL'
BASELINE_NAME = 'EXPERIMENT_1_320'
IMAGE_SIZE, BATCH_SIZE = 320, 16
BATCH_SIZE_CANDIDATES = (16, 8, 4)
AMP_ENABLED = True
EPOCHS, PATIENCE = NUM_EPOCHS, 5
CLASS_NAMES = [IDX_TO_CLASS[i] for i in range(NUM_CLASSES)]
NUM_ORDINAL_THRESHOLDS = NUM_CLASSES - 1

CHECKPOINT_PATH = Path(CHECKPOINT_DIR) / 'exp4_ordinal_best.pt'
METRICS_PATH = Path(RESULTS_METRICS_DIR) / 'exp4_ordinal_metrics.json'
HISTORY_PATH = Path(RESULTS_METRICS_DIR) / 'exp4_ordinal_history.json'
FIGURE_DIR = Path(RESULTS_FIGURES_DIR)
CONFUSION_MATRIX_PATH = FIGURE_DIR / 'exp4_ordinal_confusion_matrix.png'
CURVES_PATH = FIGURE_DIR / 'exp4_ordinal_training_curves.png'

assert IMAGE_SIZE == 320 and LEARNING_RATE == 1e-4 and WEIGHT_DECAY == 1e-4
assert EPOCHS == 20 and PATIENCE == 5
assert NUM_CLASSES == 3 and NUM_ORDINAL_THRESHOLDS == 2
assert 'exp4_ordinal' in CHECKPOINT_PATH.name
print({'seed': SEED, 'image_size': IMAGE_SIZE, 'requested_batch_size': BATCH_SIZE,
       'amp': AMP_ENABLED, 'learning_rate': LEARNING_RATE, 'optimizer': 'AdamW',
       'weight_decay': WEIGHT_DECAY, 'epochs': EPOCHS, 'patience': PATIENCE,
       'training_objective': 'BCEWithLogitsLoss on cumulative ordinal targets',
       'checkpoint_criterion': 'lowest validation ordinal loss',
       'test_evaluated': False})

## Unchanged Experiment 1 data pipeline

The transforms and official train/validation split are identical to Experiment 1. The test split is not wrapped in a dataset or loader and is not evaluated.

In [ ]:
train_transform_320 = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
eval_transform_320 = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
raw_dataset = load_dataset('moondream/glaucoma-detection')
assert [len(raw_dataset[name]) for name in ('train', 'validation')] == [2847, 1259]

def build_loaders(batch_size):
    train = GlaucomaDataset(raw_dataset['train'], transform=train_transform_320)
    val = GlaucomaDataset(raw_dataset['validation'], transform=eval_transform_320)
    return (DataLoader(train, batch_size=batch_size, shuffle=True),
            DataLoader(val, batch_size=batch_size, shuffle=False))

def verify_loaders(train_loader, val_loader):
    assert [len(x.dataset) for x in (train_loader, val_loader)] == [2847, 1259]
    images, labels = next(iter(train_loader))
    print('Image batch:', tuple(images.shape))
    print('Label batch:', tuple(labels.shape))
    assert images.shape[1:] == (3, 320, 320)
    assert labels.min() >= 0 and labels.max() < NUM_CLASSES

train_loader, val_loader = build_loaders(BATCH_SIZE)
verify_loaders(train_loader, val_loader)

## ImageNet-pretrained ResNet-50 with a two-threshold ordinal head

The existing wrapper is reused locally with two outputs representing P(y > 0) and P(y > 1). All parameters are fine-tuned with the same single AdamW learning rate as Experiment 1. The loss is unweighted BCEWithLogitsLoss over cumulative binary targets; no scheduler, class weighting, label smoothing, TTA, or differential learning rate is used.

In [ ]:
def encode_ordinal_targets(labels):
    thresholds = torch.arange(NUM_ORDINAL_THRESHOLDS, device=labels.device)
    return (labels.unsqueeze(1) > thresholds).to(torch.float32)

def decode_ordinal_logits(logits):
    assert logits.ndim == 2 and logits.shape[1] == NUM_ORDINAL_THRESHOLDS
    return (torch.sigmoid(logits) >= 0.5).sum(dim=1)

def build_training_objects():
    model = ResNet50Classifier(
        num_classes=NUM_ORDINAL_THRESHOLDS, pretrained=True, freeze_features=False).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    assert model.model.fc.in_features == 2048
    assert model.model.fc.out_features == NUM_ORDINAL_THRESHOLDS
    return model, optimizer

loss_fn = nn.BCEWithLogitsLoss()

# Deterministic encoding/decoding verification before training.
example_labels = torch.tensor([0, 1, 2], device=DEVICE)
expected_targets = torch.tensor([[0., 0.], [1., 0.], [1., 1.]], device=DEVICE)
example_logits = torch.tensor([[-2., -2.], [2., -2.], [2., 2.]], device=DEVICE)
assert torch.equal(encode_ordinal_targets(example_labels), expected_targets)
assert torch.equal(decode_ordinal_logits(example_logits), example_labels)
print('Ordinal check passed:',
      encode_ordinal_targets(example_labels).cpu().tolist(), '->',
      decode_ordinal_logits(example_logits).cpu().tolist())

probe_model, probe_optimizer = build_training_objects()
assert len(probe_optimizer.param_groups) == 1
assert probe_optimizer.param_groups[0]['lr'] == LEARNING_RATE
print(f'Trainable parameters: {sum(p.numel() for p in probe_model.parameters() if p.requires_grad):,}')
del probe_model, probe_optimizer
torch.cuda.empty_cache()

## Training

Training minimizes cumulative ordinal BCE. Early stopping and checkpoint selection use validation ordinal loss only; decoded Accuracy, Balanced Accuracy, Macro Precision, Macro Recall, Macro F1, and QWK are recorded at every validation epoch.

In [ ]:
def compute_metrics(labels, predictions):
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, labels=range(NUM_CLASSES), average='macro', zero_division=0)
    return {'accuracy': float(accuracy_score(labels, predictions)),
            'balanced_accuracy': float(balanced_accuracy_score(labels, predictions)),
            'macro_precision': float(precision), 'macro_recall': float(recall),
            'macro_f1': float(f1),
            'qwk': float(cohen_kappa_score(labels, predictions, weights='quadratic'))}

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss, predictions, labels_all = 0.0, [], []
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        logits = model(images)
        loss = loss_fn(logits, encode_ordinal_targets(labels))
        total_loss += loss.item() * images.size(0)
        predictions.extend(decode_ordinal_logits(logits).cpu().tolist())
        labels_all.extend(labels.cpu().tolist())
    metrics = compute_metrics(labels_all, predictions)
    metrics['loss'] = float(total_loss / len(loader.dataset))
    return metrics, predictions, labels_all

def fit(model, optimizer, train_loader, val_loader):
    history = {'train_loss': [], 'val_loss': [], 'val_metrics': []}
    scaler = torch.amp.GradScaler('cuda', enabled=AMP_ENABLED)
    best_loss, best_epoch, best_metrics, stale = float('inf'), None, None, 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        total_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            with torch.autocast('cuda', dtype=torch.float16, enabled=AMP_ENABLED):
                logits = model(images)
                loss = loss_fn(logits, encode_ordinal_targets(labels))
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * images.size(0)
        train_loss = total_loss / len(train_loader.dataset)
        metrics, _, _ = evaluate(model, val_loader)
        history['train_loss'].append(float(train_loss))
        history['val_loss'].append(metrics['loss'])
        history['val_metrics'].append(metrics)
        improved = metrics['loss'] < best_loss
        print(f"Epoch {epoch:02d}/{EPOCHS} | Train Loss {train_loss:.4f} | "
              f"Validation Loss {metrics['loss']:.4f} | Accuracy {metrics['accuracy']:.4f} | "
              f"Balanced Accuracy {metrics['balanced_accuracy']:.4f} | "
              f"Macro Precision {metrics['macro_precision']:.4f} | "
              f"Macro Recall {metrics['macro_recall']:.4f} | "
              f"Macro F1 {metrics['macro_f1']:.4f} | QWK {metrics['qwk']:.4f}"
              + (' (best validation loss)' if improved else ''))
        if improved:
            best_loss, best_epoch, best_metrics, stale = metrics['loss'], epoch, metrics.copy(), 0
            CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
            torch.save(model.state_dict(), CHECKPOINT_PATH)
        else:
            stale += 1
        if stale >= PATIENCE:
            print(f'No validation-loss improvement for {PATIENCE} epochs; stopping.')
            break
    return history, best_epoch, best_metrics

oom_occurred = False
for batch_size in BATCH_SIZE_CANDIDATES:
    gc.collect(); torch.cuda.empty_cache(); set_seed(SEED)
    train_loader, val_loader = build_loaders(batch_size)
    print(f'\nAttempting Experiment 4 with batch size {batch_size}')
    verify_loaders(train_loader, val_loader)
    set_seed(SEED)
    model, optimizer = build_training_objects()
    attempt_started = time.perf_counter()
    try:
        history, best_epoch, best_val_metrics = fit(model, optimizer, train_loader, val_loader)
        training_time_seconds = time.perf_counter() - attempt_started
        actual_batch_size = batch_size
        break
    except RuntimeError as error:
        if 'out of memory' not in str(error).lower(): raise
        oom_occurred = True
        print(f'CUDA OOM at batch size {batch_size}; restarting from ImageNet initialization at epoch 1.')
        del model, optimizer
else:
    raise RuntimeError('CUDA OOM at batch sizes 16, 8, and 4')

assert CHECKPOINT_PATH.exists()
print({'epochs_completed': len(history['train_loss']), 'best_epoch': best_epoch,
       'best_validation_loss': best_val_metrics['loss'], 'actual_batch_size': actual_batch_size,
       'learning_rate': LEARNING_RATE, 'training_time_seconds': training_time_seconds,
       'amp': AMP_ENABLED})

## Experiment 4 validation histories

In [ ]:
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
epochs_ran = list(range(1, len(history['train_loss']) + 1))
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
axes[0, 0].plot(epochs_ran, history['train_loss'], marker='o', label='Train')
axes[0, 0].plot(epochs_ran, history['val_loss'], marker='o', label='Validation')
axes[0, 0].set(title='Ordinal BCE loss', xlabel='Epoch'); axes[0, 0].legend()
for key, label in [('accuracy', 'Accuracy'), ('balanced_accuracy', 'Balanced accuracy')]:
    axes[0, 1].plot(epochs_ran, [m[key] for m in history['val_metrics']], marker='o', label=label)
axes[0, 1].set(title='Validation accuracy', xlabel='Epoch'); axes[0, 1].legend()
for key, label in [('macro_precision', 'Macro precision'), ('macro_recall', 'Macro recall'),
                   ('macro_f1', 'Macro F1')]:
    axes[1, 0].plot(epochs_ran, [m[key] for m in history['val_metrics']], marker='o', label=label)
axes[1, 0].set(title='Validation macro metrics', xlabel='Epoch'); axes[1, 0].legend()
axes[1, 1].plot(epochs_ran, [m['qwk'] for m in history['val_metrics']], marker='o')
axes[1, 1].set(title='Validation QWK', xlabel='Epoch')
fig.suptitle('Experiment 4: ResNet-50 320x320 ordinal regression')
fig.tight_layout(); fig.savefig(CURVES_PATH, dpi=160); plt.show()

## Selected-checkpoint validation evaluation

This reloads only the checkpoint produced by Experiment 4, then reports validation metrics, per-class results, all six directed severity-error counts, and the confusion matrix. Ground truth is shown by rows and predictions by columns.

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=True))
selected_val_metrics, val_predictions, val_labels = evaluate(model, val_loader)
assert abs(selected_val_metrics['loss'] - best_val_metrics['loss']) < 1e-8
report_dict = classification_report(
    val_labels, val_predictions, labels=range(NUM_CLASSES), target_names=CLASS_NAMES,
    digits=4, zero_division=0, output_dict=True)
per_class_metrics = {
    name: {key: float(report_dict[name][key]) for key in ('precision', 'recall', 'f1-score')}
    for name in CLASS_NAMES
}
print(json.dumps(selected_val_metrics, indent=2))
print(classification_report(val_labels, val_predictions, labels=range(NUM_CLASSES),
      target_names=CLASS_NAMES, digits=4, zero_division=0))
print('Per-class metrics:', json.dumps(per_class_metrics, indent=2))

matrix = confusion_matrix(val_labels, val_predictions, labels=range(NUM_CLASSES))
ordinal_errors = {
    'Normal -> Early': int(matrix[0, 1]), 'Normal -> Advanced': int(matrix[0, 2]),
    'Early -> Normal': int(matrix[1, 0]), 'Early -> Advanced': int(matrix[1, 2]),
    'Advanced -> Early': int(matrix[2, 1]), 'Advanced -> Normal': int(matrix[2, 0]),
}
print('Directed ordinal errors:', json.dumps(ordinal_errors, indent=2))

plt.figure(figsize=(7, 6))
sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel('Predicted'); plt.ylabel('Ground truth')
plt.title('Experiment 4 validation confusion matrix'); plt.tight_layout()
plt.savefig(CONFUSION_MATRIX_PATH, dpi=160); plt.show()

## Save Experiment 4 artifacts and compare with Experiment 1

The comparison is validation-only and follows the predefined decision rule. Experiment 1 aggregate metrics are read from its saved metrics file when available, with the supplied reference values as fallback. Missing Experiment 1 class-level, confusion-matrix, or history data is reported rather than fabricated.

In [ ]:
METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
history_record = {
    'experiment': EXPERIMENT_NAME, 'seed': SEED, 'image_size': IMAGE_SIZE,
    'batch_size': actual_batch_size, 'learning_rate': LEARNING_RATE,
    'training_objective': 'BCEWithLogitsLoss on cumulative ordinal targets',
    'checkpoint_criterion': 'lowest validation ordinal loss', 'history': history,
}
with HISTORY_PATH.open('w', encoding='utf-8') as file:
    json.dump(history_record, file, indent=2)

exp1_reference = {
    'accuracy': 0.9031, 'macro_f1': 0.8922, 'qwk': 0.9035, 'loss': 0.2648
}
exp1_path = Path(RESULTS_METRICS_DIR) / 'resnet50_320.json'
exp1_results = {}
if exp1_path.exists():
    with exp1_path.open(encoding='utf-8') as file: exp1_results = json.load(file)
exp1_metrics = {**exp1_reference, **exp1_results.get('validation_metrics', {})}

rows = [('Validation loss', 'loss'), ('Accuracy', 'accuracy'),
        ('Balanced Accuracy', 'balanced_accuracy'), ('Macro F1', 'macro_f1'), ('QWK', 'qwk')]
print('\nMetric'.ljust(26), BASELINE_NAME.rjust(20), EXPERIMENT_NAME.rjust(24), 'Delta'.rjust(12))
print('-' * 86)
for label, key in rows:
    old = exp1_metrics.get(key)
    old_text = f'{old:.4f}' if old is not None else 'not available'
    delta_text = f"{selected_val_metrics[key] - old:+.4f}" if old is not None else 'n/a'
    print(label.ljust(26), old_text.rjust(20),
          f"{selected_val_metrics[key]:.4f}".rjust(24), delta_text.rjust(12))
print('Note: Experiment 1 cross-entropy loss and Experiment 4 ordinal BCE have different scales; '
      'both are reported, but their numerical delta is not a like-for-like quality measure.')

exp1_report = exp1_results.get('classification_report', {})
if 'early' in exp1_report:
    print('\nEarly-class comparison:')
    for key in ('precision', 'recall', 'f1-score'):
        old, new = float(exp1_report['early'][key]), per_class_metrics['early'][key]
        print(f'{key}: Exp1={old:.4f}, Exp4={new:.4f}, delta={new - old:+.4f}')
else:
    print('\nExperiment 1 Early precision/recall/F1 are unavailable in its saved metrics; '
          'Experiment 4 values are reported without fabricating a baseline.')

for key, label in [('confusion_matrix', 'confusion matrix'), ('history', 'training/validation dynamics')]:
    if key not in exp1_results:
        print(f'Experiment 1 {label} unavailable in its saved metrics; inspect its original artifacts for comparison.')

metric_deltas = {key: selected_val_metrics[key] - exp1_metrics[key]
                 for key in ('loss', 'accuracy', 'balanced_accuracy', 'macro_f1', 'qwk')
                 if key in exp1_metrics}
decision_deltas = {key: metric_deltas[key]
                   for key in ('accuracy', 'balanced_accuracy', 'macro_f1', 'qwk')
                   if key in metric_deltas}
if decision_deltas and all(delta < 0 for delta in decision_deltas.values()):
    decision = 'Reject Experiment 4: all available major validation metrics decreased.'
elif metric_deltas['qwk'] < 0:
    decision = 'Do not select Experiment 4: QWK decreased, regardless of any Accuracy gain.'
elif metric_deltas['macro_f1'] >= 0:
    decision = ('Experiment 4 is promising on aggregate validation metrics; confirm the magnitude of the QWK gain, '
                'Early-class performance, and severity-error pattern before declaring it better.')
else:
    decision = ('Experiment 4 improves QWK but lowers Macro-F1; treat this as a severity-aware trade-off '
                'and inspect Early-class and directed-error results.')
print('\nDecision:', decision)

results = {
    'experiment': EXPERIMENT_NAME, 'baseline': BASELINE_NAME,
    'description': 'Experiment 4 — Ordinal Regression / Ordinal Loss',
    'hypothesis': ('An ordinal objective may better model glaucoma severity, reduce severe ordinal '
                   'misclassifications, and improve QWK and Early-class performance.'),
    'controlled_change': ('Three-class CrossEntropyLoss replaced by two-threshold cumulative '
                          'ordinal regression with BCEWithLogitsLoss.'),
    'model': 'ResNet50Classifier(num_classes=2, pretrained=True, freeze_features=False)',
    'initialization': 'torchvision ImageNet pretrained weights; independent run with a new two-logit head',
    'ordinal_targets': {'normal': [0, 0], 'early': [1, 0], 'advanced': [1, 1]},
    'decoding': 'sigmoid(logits) >= 0.5 for each threshold; predicted class is satisfied-threshold count',
    'class_mapping': {str(i): CLASS_NAMES[i] for i in range(NUM_CLASSES)},
    'seed': SEED, 'image_size': IMAGE_SIZE, 'batch_size': actual_batch_size,
    'amp_enabled': AMP_ENABLED, 'cuda_oom_occurred': oom_occurred,
    'epochs_completed': len(history['train_loss']), 'maximum_epochs': EPOCHS,
    'early_stopping_patience': PATIENCE, 'best_epoch': best_epoch,
    'best_validation_loss': selected_val_metrics['loss'],
    'training_objective': 'BCEWithLogitsLoss on cumulative ordinal targets',
    'checkpoint_criterion': 'lowest validation ordinal loss',
    'optimizer': 'AdamW', 'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY, 'scheduler': None,
    'training_time_seconds': training_time_seconds,
    'validation_metrics': selected_val_metrics,
    'per_class_metrics': per_class_metrics,
    'classification_report': report_dict, 'confusion_matrix': matrix.tolist(),
    'directed_ordinal_errors': ordinal_errors,
    'experiment_1_validation_metrics': exp1_metrics,
    'metric_deltas_vs_experiment_1': metric_deltas,
    'loss_comparison_note': ('Cross-entropy and ordinal BCE have different scales; '
                             'the loss delta is descriptive, not directly comparable.'),
    'decision': decision, 'test_evaluated': False,
    'artifacts': {'checkpoint': str(CHECKPOINT_PATH), 'metrics': str(METRICS_PATH),
                  'history': str(HISTORY_PATH), 'confusion_matrix': str(CONFUSION_MATRIX_PATH),
                  'training_curves': str(CURVES_PATH)},
}
with METRICS_PATH.open('w', encoding='utf-8') as file:
    json.dump(results, file, indent=2)
print('Saved Experiment 4 artifacts:', CHECKPOINT_PATH, METRICS_PATH, HISTORY_PATH,
      CONFUSION_MATRIX_PATH, CURVES_PATH, sep='\n')